<a href="https://colab.research.google.com/github/sofiamiiy/Serpientes/blob/main/Copia_de_Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"Serpientes y escaleras"

El juego incluye cambios tales como:

Escaleras: 3 → 11; 15 → 19

Serpientes: 13 → 4; 17 → 10

Se usa un dado de 6 caras

# Modelo probabilístico

Sea \(X_n\) la posición del jugador después de la tirada \(n\).

La cadena de Markov cumple:

$$
P_{ij} = P(X_{n+1}=j \mid X_n=i)
$$

donde:

- \(i\) representa la casilla actual.
- \(j\) representa la siguiente casilla.
- \(P_{ij}\) es la probabilidad de transición entre estados.

Como el dado es justo:

$$
P(\text{avanzar } k \text{ casillas}) = \frac{1}{6}
$$

para:

$$
k \in \{1,2,3,4,5,6\}
$$

In [1]:
# Importar librerías y crear tablero
import random
import numpy as np
import sympy as sp
from sympy import Matrix, Rational

cambios = {
    3: 11,   # escalera
    15: 19,  # escalera
    13: 4,   # serpiente
    17: 10   # serpiente
}

N = 20  # número de casillas

# Matriz de transición

Construimos la matriz de transición \(P\).

Cada entrada de la matriz representa la probabilidad de pasar de una casilla a otra en una sola tirada.

La matriz tiene la forma:

$$
P =
\begin{pmatrix}
P_{11} & P_{12} & \cdots & P_{1n} \\
P_{21} & P_{22} & \cdots & P_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
P_{n1} & P_{n2} & \cdots & P_{nn}
\end{pmatrix}
$$

La casilla 20 es absorbente, por lo que:

$$
P_{20,20}=1
$$

In [2]:
# Crear matriz de transición vacía
P = sp.zeros(N, N)

for i in range(N):

    # Estado final absorbente
    if i == N - 1:
        P[i, i] = 1
        continue

    # Posición real en tablero (1-20)
    posicion = i + 1

    for dado in range(1, 7):

        nueva = posicion + dado

        # Si se pasa de 20, permanece en 20
        if nueva > N:
            nueva = N

        # Aplicar serpientes y escaleras
        if nueva in cambios:
            nueva = cambios[nueva]

        # Convertir a índice
        j = nueva - 1

        P[i, j] += Rational(1, 6)

P

Matrix([
[0, 1/6, 0, 1/6, 1/6, 1/6, 1/6,   0,   0,   0, 1/6,   0, 0,   0, 0,   0, 0,   0,   0,   0],
[0,   0, 0, 1/6, 1/6, 1/6, 1/6, 1/6,   0,   0, 1/6,   0, 0,   0, 0,   0, 0,   0,   0,   0],
[0,   0, 0, 1/6, 1/6, 1/6, 1/6, 1/6, 1/6,   0,   0,   0, 0,   0, 0,   0, 0,   0,   0,   0],
[0,   0, 0,   0, 1/6, 1/6, 1/6, 1/6, 1/6, 1/6,   0,   0, 0,   0, 0,   0, 0,   0,   0,   0],
[0,   0, 0,   0,   0, 1/6, 1/6, 1/6, 1/6, 1/6, 1/6,   0, 0,   0, 0,   0, 0,   0,   0,   0],
[0,   0, 0,   0,   0,   0, 1/6, 1/6, 1/6, 1/6, 1/6, 1/6, 0,   0, 0,   0, 0,   0,   0,   0],
[0,   0, 0, 1/6,   0,   0,   0, 1/6, 1/6, 1/6, 1/6, 1/6, 0,   0, 0,   0, 0,   0,   0,   0],
[0,   0, 0, 1/6,   0,   0,   0,   0, 1/6, 1/6, 1/6, 1/6, 0, 1/6, 0,   0, 0,   0,   0,   0],
[0,   0, 0, 1/6,   0,   0,   0,   0,   0, 1/6, 1/6, 1/6, 0, 1/6, 0,   0, 0,   0, 1/6,   0],
[0,   0, 0, 1/6,   0,   0,   0,   0,   0,   0, 1/6, 1/6, 0, 1/6, 0, 1/6, 0,   0, 1/6,   0],
[0,   0, 0, 1/6,   0,   0,   0,   0,   0, 1/6,   0, 1/6, 0, 1/6, 0, 1/6

# Cadena de Markov absorbente

La matriz de transición puede escribirse como:

$$
P =
\begin{pmatrix}
Q & R \\
0 & I
\end{pmatrix}
$$

donde:

- \(Q\) probabilidades entre estados transientes.
- \(R\) probabilidades hacia estados absorbentes.
- \(I\) representa el estado absorbente.

Para calcular el número esperado de tiradas se utiliza la matriz fundamental:

$$
F = (I-Q)^{-1}
$$

In [3]:
# Matriz Q (todos menos el estado absorbente)
Q = P[:N-1, :N-1]

# Matriz identidad
I = sp.eye(N-1)

# Matriz fundamental
F = (I - Q).inv()

F

Matrix([
[1, 1/6, 0, 20157725/30132222, 1536444/5022037, 1792518/5022037, 2091271/5022037, 1602810/5022037, 62295983/180793332, 127748335/180793332, 18161095/30132222, 13792457/30132222, 0, 73244465/180793332, 0, 16363088/45198333, 0, 221451559/1084759992, 3783693031/6508559952],
[0,   1, 0,   3251627/5022037, 1378944/5022037, 1608768/5022037, 1876896/5022037, 2189712/5022037,  10305947/30132222,   21111211/30132222,   2885425/5022037,   2299499/5022037, 0,  12610829/30132222, 0,  5402632/15066111, 0,   37213087/180793332,  628534831/1084759992],
[0,   0, 1,   3296987/5022037, 1386504/5022037, 1617588/5022037, 1887186/5022037, 2201717/5022037,  15412019/30132222, 149486245/210925554, 15424338/35154259,   2339639/5022037, 0,  12872699/30132222, 0,  5123881/15066111, 0,   37158295/180793332,  637272607/1084759992],
[0,   0, 0,   7410528/5022037, 1235088/5022037, 1440936/5022037, 1681092/5022037, 1961274/5022037,    2288153/5022037,     4039609/5022037,   2107692/5022037,   2253126/502203

# Tiempo esperado de absorción

El vector de tiempos esperados se calcula mediante:

$$
t = F\mathbf{1}
$$

donde:

$$
\mathbf{1} =
\begin{pmatrix}
1 \\
1 \\
\vdots \\
1
\end{pmatrix}
$$

Cada entrada \(t_i\) representa el número esperado de tiradas para llegar a la casilla final iniciando desde el estado \(i\).

In [4]:
# Vector de unos
unos = sp.ones(N-1, 1)

# Tiempos esperados
t = F * unos

t

#Numero promedio de tiradas desde la casila 1
esperanza = sp.N(t[0], 6)

print("Número promedio de tiradas para terminar el juego:")
print(esperanza)




Número promedio de tiradas para terminar el juego:
6.89835


# Simulación Monte Carlo

Ahora realizamos una simulación del juego para aproximar experimentalmente el número promedio de tiradas.

La idea consiste en:

1. Simular muchas partidas.
2. Contar el número de tiradas en cada partida.
3. Calcular el promedio experimental.

In [8]:
def jugar():

    posicion = 1
    tiradas = 0

    while posicion < 20:

        dado = random.randint(1, 6)
        posicion += dado

        if posicion > 20:
            posicion = 20

        # Aplicar serpientes y escaleras
        if posicion in cambios:
            posicion = cambios[posicion]

        tiradas += 1
    return tiradas

#Ejecutar simulaciones

num_simulaciones = 100000

resultados = [jugar() for _ in range(num_simulaciones)]

promedio_simulado = np.mean(resultados)

print("Promedio simulado de tiradas:")
print(promedio_simulado)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 23)

# Conclusión

Se modeló el juego mediante cadenas de Markov.

Las ecuaciones principales utilizadas fueron:

$$
P_{ij} = P(X_{n+1}=j \mid X_n=i)
$$

$$
F = (I-Q)^{-1}
$$

$$
t = F\mathbf{1}
$$

El valor teórico obtenido mediante álgebra matricial puede compararse con el promedio obtenido experimentalmente usando simulación Monte Carlo.

# Ratón buscando comida

Modelaremos el movimiento aleatorio de un ratón en un tablero 3x3 usando una cadena de Markov.


$$
\begin{matrix}
0 & 1 & 7 \\
2 & 3 & 4 \\
8 & 5 & 6
\end{matrix}
$$

donde:

- La casilla \(7\) contiene comida.
- La casilla \(8\) contiene una trampa.


# Modelo probabilístico

Sea \(X_n\) la posición del ratón después del movimiento \(n\).

La cadena de Markov satisface:

$$
P_{ij} = P(X_{n+1}=j \mid X_n=i)
$$

donde:

- \(i\) es la casilla actual.
- \(j\) es la casilla siguiente.

En cada paso el ratón elige una casilla vecina válida.

#Importar librerías y definir tablero

In [ ]:
import random
import numpy as np
import sympy as sp
from sympy import Matrix, Rational

# Tablero
tablero = [
    [0, 1, 7],
    [2, 3, 4],
    [8, 5, 6]
]

# Estados absorbentes
comida = 7
trampa = 8

# Coordenadas de cada estado
posiciones = {
    0:(0,0),
    1:(0,1),
    7:(0,2),
    2:(1,0),
    3:(1,1),
    4:(1,2),
    8:(2,0),
    5:(2,1),
    6:(2,2)
}

estados = [0,1,2,3,4,5,6,7,8]

N = len(estados)

# Movimientos posibles

El ratón puede moverse únicamente hacia:

arriba, abajo, izquierda, derecha

siempre que la casilla exista dentro del tablero.

La probabilidad de transición depende del número de vecinos disponibles.

In [ ]:
#definir estados vecinos
def vecinos(estado):

    fila, col = posiciones[estado]

    movimientos = [
        (-1,0),  # arriba
        (1,0),   # abajo
        (0,-1),  # izquierda
        (0,1)    # derecha
    ]

    resultado = []

    for df, dc in movimientos:

        nf = fila + df
        nc = col + dc

        if 0 <= nf < 3 and 0 <= nc < 3:
            resultado.append(tablero[nf][nc])

    return resultado

# Construcción de la matriz de transición

La matriz de transición tiene la forma:

$$
P =
\begin{pmatrix}
P_{11} & P_{12} & \cdots & P_{1n} \\
P_{21} & P_{22} & \cdots & P_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
P_{n1} & P_{n2} & \cdots & P_{nn}
\end{pmatrix}
$$

Los estados \(7\) y \(8\) son absorbentes:

$$
P_{77}=1
$$

$$
P_{88}=1
$$

In [ ]:
# Matriz de transición
P = sp.zeros(N, N)

for i, estado in enumerate(estados):

    # Estados absorbentes
    if estado in [7,8]:
        P[i,i] = 1
        continue

    vecinos_estado = vecinos(estado)

    prob = Rational(1, len(vecinos_estado))

    for v in vecinos_estado:

        j = estados.index(v)

        P[i,j] += prob

P

# Cadena absorbente

La matriz puede escribirse como:

$$
P =
\begin{pmatrix}
Q & R \\
0 & I
\end{pmatrix}
$$

donde:

\(Q\): transiciones entre estados transientes.

\(R\): transiciones hacia estados absorbentes.

La matriz fundamental es:

$$
F = (I-Q)^{-1}
$$

y permite calcular:

- tiempos esperados,
- probabilidades de absorción.

In [ ]:
# Estados transientes
transientes = [0,1,2,3,4,5,6]

# Índices
idx_trans = [estados.index(x) for x in transientes]

# Submatriz Q
Q = P.extract(idx_trans, idx_trans)

# Identidad
I = sp.eye(len(Q))

# Matriz fundamental
F = (I - Q).inv()

F

# Tiempo esperado hasta absorción

El vector de tiempos esperados se calcula mediante:

$$
t = F\mathbf{1}
$$


In [ ]:
# Vector de unos
unos = sp.ones(len(Q),1)

# Tiempos esperados
t = F * unos

t

# Probabilidades de llegar a la comida o la trampa

Las probabilidades de absorción se calculan mediante:

$$
B = FR
$$

donde:

- \(F\) es la matriz fundamental.
- \(R\) contiene las transiciones hacia estados absorbentes.

Cada entrada de \(B\) representa la probabilidad de terminar en un estado absorbente específico.

In [ ]:
# Índices absorbentes
idx_abs = [estados.index(7), estados.index(8)]

# Matriz R
R = P.extract(idx_trans, idx_abs)

# Probabilidades de absorción
B = F * R

B

# Simulación Monte Carlo

Ahora simularemos muchas trayectorias aleatorias del ratón para comparar los resultados teóricos con resultados experimentales.

In [ ]:
def simular():

    estado = 0
    pasos = 0

    while estado not in [7,8]:

        estado = random.choice(vecinos(estado))

        pasos += 1

    return estado, pasos

    num_simulaciones = 100000

comida_count = 0
trampa_count = 0
pasos_totales = 0

for _ in range(num_simulaciones):

    estado_final, pasos = simular()

    pasos_totales += pasos

    if estado_final == 7:
        comida_count += 1
    else:
        trampa_count += 1

print("Probabilidad de llegar a comida:")
print(comida_count / num_simulaciones)

print()

print("Probabilidad de caer en trampa:")
print(trampa_count / num_simulaciones)

print()

print("Promedio de pasos:")
print(pasos_totales / num_simulaciones)

